In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

# 1. Load Dataset

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target, name='target')
target_names = wine.target_names  # ['class_0', 'class_1', 'class_2']

print("=" * 60)
print("          LAB TASK 1: WINE CLASSIFICATION")
print("=" * 60)
print(f"\n Dataset Shape  : {X.shape}")
print(f" Features       : {X.shape[1]}")
print(f" Classes        : {list(target_names)}")
print(f" Class counts   :\n{y.value_counts().rename({0:'class_0',1:'class_1',2:'class_2'})}")

print("\n── First 5 rows ──")
print(X.head())
print("\n── Basic Statistics ──")
print(X.describe().round(2))

# 2. Train / Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"\n Train samples : {len(X_train)}")
print(f" Test  samples : {len(X_test)}")

# 3a. Gaussian Naive Bayes

gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred_gnb = gnb.predict(X_test)
acc_gnb = accuracy_score(y_test, y_pred_gnb)

print("\n" + "=" * 60)
print("  GAUSSIAN NAIVE BAYES")
print("=" * 60)
print(f"  Accuracy : {acc_gnb:.4f}  ({acc_gnb*100:.2f}%)")
print("\n  Classification Report:")
print(classification_report(y_test, y_pred_gnb, target_names=target_names))

# 3b. Multinomial Naive Bayes
#     (requires non-negative features → MinMaxScaler)

scaler = MinMaxScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

mnb = MultinomialNB()
mnb.fit(X_train_sc, y_train)
y_pred_mnb = mnb.predict(X_test_sc)
acc_mnb = accuracy_score(y_test, y_pred_mnb)

print("\n" + "=" * 60)
print("  MULTINOMIAL NAIVE BAYES")
print("=" * 60)
print(f"  Accuracy : {acc_mnb:.4f}  ({acc_mnb*100:.2f}%)")
print("\n  Classification Report:")
print(classification_report(y_test, y_pred_mnb, target_names=target_names))

# 4. Comparison

print("\n" + "=" * 60)
print("  MODEL COMPARISON")
print("=" * 60)
print(f"  Gaussian NB   Accuracy : {acc_gnb*100:.2f}%")
print(f"  Multinomial NB Accuracy: {acc_mnb*100:.2f}%")
winner = "Gaussian NB" if acc_gnb >= acc_mnb else "Multinomial NB"
print(f"\n  ✔  Better Model: {winner}")

# 5. Predictions on Test Data (first 10)

print("\n" + "=" * 60)
print("  SAMPLE PREDICTIONS (first 10 test samples)")
print("=" * 60)
pred_df = pd.DataFrame({
    'Actual'        : [target_names[i] for i in y_test[:10]],
    'GaussianNB'    : [target_names[i] for i in y_pred_gnb[:10]],
    'MultinomialNB' : [target_names[i] for i in y_pred_mnb[:10]],
})
pred_df['GNB_Correct']  = pred_df['Actual'] == pred_df['GaussianNB']
pred_df['MNB_Correct']  = pred_df['Actual'] == pred_df['MultinomialNB']
print(pred_df.to_string(index=True))

# 6. Visualizations

fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor('#0f1117')

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

COLORS = ['#4FC3F7', '#81C784', '#FFB74D']
ACCENT = '#E040FB'

# -- (A) Class Distribution
ax0 = fig.add_subplot(gs[0, 0])
counts = [sum(y == i) for i in range(3)]
bars = ax0.bar(target_names, counts, color=COLORS, edgecolor='white', linewidth=0.6)
ax0.set_facecolor('#1a1d27'); ax0.tick_params(colors='white')
ax0.set_title('Class Distribution', color='white', fontsize=11, fontweight='bold')
ax0.set_ylabel('Count', color='#aaa')
for bar, cnt in zip(bars, counts):
    ax0.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, str(cnt),
             ha='center', color='white', fontsize=10)
ax0.spines[:].set_color('#333')

# -- (B) Accuracy Bar
ax1 = fig.add_subplot(gs[0, 1])
models  = ['Gaussian NB', 'Multinomial NB']
accs    = [acc_gnb*100, acc_mnb*100]
clrs    = ['#4FC3F7', '#FFB74D']
b2 = ax1.bar(models, accs, color=clrs, edgecolor='white', linewidth=0.6)
ax1.set_ylim(0, 110)
ax1.set_facecolor('#1a1d27'); ax1.tick_params(colors='white')
ax1.set_title('Model Accuracy (%)', color='white', fontsize=11, fontweight='bold')
for bar, a in zip(b2, accs):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             f'{a:.1f}%', ha='center', color='white', fontsize=11, fontweight='bold')
ax1.spines[:].set_color('#333')

# -- (C) Feature Importance (variance)
ax2 = fig.add_subplot(gs[0, 2])
variances = X.var().sort_values(ascending=True)
ax2.barh(variances.index, variances.values, color='#AB47BC', edgecolor='white', linewidth=0.4)
ax2.set_facecolor('#1a1d27'); ax2.tick_params(colors='white', labelsize=7)
ax2.set_title('Feature Variance', color='white', fontsize=11, fontweight='bold')
ax2.spines[:].set_color('#333')

# -- (D) Confusion Matrix — Gaussian NB
ax3 = fig.add_subplot(gs[1, 0:2])
cm_gnb = confusion_matrix(y_test, y_pred_gnb)
disp = ConfusionMatrixDisplay(cm_gnb, display_labels=target_names)
disp.plot(ax=ax3, colorbar=False, cmap='Blues')
ax3.set_title('Confusion Matrix — Gaussian NB', color='white', fontsize=11, fontweight='bold')
ax3.set_facecolor('#1a1d27'); ax3.tick_params(colors='white')
ax3.xaxis.label.set_color('white'); ax3.yaxis.label.set_color('white')
for text in ax3.texts: text.set_color('white')

# -- (E) Confusion Matrix — Multinomial NB
ax4 = fig.add_subplot(gs[1, 2])
cm_mnb = confusion_matrix(y_test, y_pred_mnb)
disp2 = ConfusionMatrixDisplay(cm_mnb, display_labels=target_names)
disp2.plot(ax=ax4, colorbar=False, cmap='Oranges')
ax4.set_title('Confusion Matrix — Multinomial NB', color='white', fontsize=10, fontweight='bold')
ax4.set_facecolor('#1a1d27'); ax4.tick_params(colors='white', labelsize=7)
ax4.xaxis.label.set_color('white'); ax4.yaxis.label.set_color('white')
for text in ax4.texts: text.set_color('white')

# -- (F) Feature Distributions per class (alcohol)
ax5 = fig.add_subplot(gs[2, 0])
for i, cl in enumerate(target_names):
    vals = X.loc[y == i, 'alcohol']
    ax5.hist(vals, bins=12, alpha=0.7, color=COLORS[i], label=cl, edgecolor='white', lw=0.3)
ax5.set_facecolor('#1a1d27'); ax5.tick_params(colors='white')
ax5.set_title('Alcohol Distribution by Class', color='white', fontsize=10, fontweight='bold')
ax5.legend(fontsize=8, facecolor='#222', labelcolor='white')
ax5.spines[:].set_color('#333')

# -- (G) Feature Distributions (flavanoids)
ax6 = fig.add_subplot(gs[2, 1])
for i, cl in enumerate(target_names):
    vals = X.loc[y == i, 'flavanoids']
    ax6.hist(vals, bins=12, alpha=0.7, color=COLORS[i], label=cl, edgecolor='white', lw=0.3)
ax6.set_facecolor('#1a1d27'); ax6.tick_params(colors='white')
ax6.set_title('Flavanoids Distribution by Class', color='white', fontsize=10, fontweight='bold')
ax6.legend(fontsize=8, facecolor='#222', labelcolor='white')
ax6.spines[:].set_color('#333')

# -- (H) GNB Prediction Probabilities (test set, first 20)
ax7 = fig.add_subplot(gs[2, 2])
probs = gnb.predict_proba(X_test[:20])
for i, cl in enumerate(target_names):
    ax7.plot(range(20), probs[:, i], 'o-', color=COLORS[i], label=cl, lw=1.5, ms=4)
ax7.set_facecolor('#1a1d27'); ax7.tick_params(colors='white')
ax7.set_title('GNB — Class Probabilities\n(first 20 test samples)', color='white', fontsize=9, fontweight='bold')
ax7.legend(fontsize=7, facecolor='#222', labelcolor='white')
ax7.set_xlabel('Sample index', color='#aaa')
ax7.spines[:].set_color('#333')

# Title
fig.suptitle('Lab Task 1 — Wine Classification | Gaussian vs Multinomial Naive Bayes',
             color='white', fontsize=14, fontweight='bold', y=0.98)

plt.savefig('lab_task1_wine.png', dpi=150,
            bbox_inches='tight', facecolor=fig.get_facecolor())
print("\n[✔] Plot saved → lab_task1_wine.png")
plt.close()


          LAB TASK 1: WINE CLASSIFICATION

 Dataset Shape  : (178, 13)
 Features       : 13
 Classes        : [np.str_('class_0'), np.str_('class_1'), np.str_('class_2')]
 Class counts   :
target
class_1    71
class_0    59
class_2    48
Name: count, dtype: int64

── First 5 rows ──
   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   
3    14.37        1.95  2.50               16.8      113.0           3.85   
4    13.24        2.59  2.87               21.0      118.0           2.80   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.04   
1        2.76                  0.26             1.28             4.38  1.05   
2        3.24   